In [67]:
import mpmath
import matplotlib.pyplot as plt
import matplotlib

mpmath.mp.dps = 50  # Set decimal places for high precision
def solve_network_log_corrected(n, m_star):
    nc2 = int(mpmath.binomial(n, 2))
    
    # Initial guess using asymptotic inversion: N ~ lnW / lnlnW
    ln_B = mpmath.log(mpmath.binomial(nc2, m_star))
    target_val = ln_B / mpmath.log(ln_B)
    
    # theta is the derivative of target_val w.r.t m
    binom_slope = mpmath.log((nc2 - m_star) / m_star)
    theta_guess = binom_slope * (1/mpmath.log(ln_B) - 1/(mpmath.log(ln_B)**2))
    alpha_guess = target_val - theta_guess * m_star

    def equations(alpha, theta):
        total_prob = 0
        expected_edges = 0
        for m in range(nc2 + 1):
            H = -(alpha + theta * m)
            log_p = -H * mpmath.re(mpmath.lambertw(1/H))
            
            prob = mpmath.binomial(nc2, m) * mpmath.exp(log_p)
            total_prob += prob
            expected_edges += m * prob
            
            # Optimization: break when probability mass becomes negligible
            if m > m_star and prob < 1e-20:
                break
        return [total_prob - 1, expected_edges - m_star]

    solvers = ['secant', 'newton', 'bisect', 'anderson', 'muller']
    for solver in solvers:
        print(f"Trying solver '{solver}' for n={n}")
        try:
            kwargs = {'tol': 1e-10}
            root = mpmath.findroot(equations, [alpha_guess, theta_guess], solver=solver, **kwargs)
            print(f"  [Success] Solver '{solver}' converged for m={m_star}")
            return float(root[0]), float(root[1])
            
        except Exception as e:
            print(f"  [Failed] Solver '{solver}' failed with error: {e}")
            continue
            
    print(f"  [Error] All solvers failed for m={m_star}")
    return None, None

def calculate_log_corrected_entropy(alpha, theta, nc2):
    total_entropy = 0
    for m in range(int(nc2) + 1):
        H = -(alpha + theta * m)
        w = mpmath.lambertw(1/H)
        if mpmath.im(w) > 1e-10:
            print("Warning: System in non-physical complex regime")
        log_p = -H * mpmath.re(w)
        p = mpmath.exp(log_p)

        term = -p * log_p / mpmath.log(-log_p)

        total_entropy += mpmath.binomial(nc2, m) * term
        """
        if m > nc2/2 and term < 1e-20:
            break"""
    return total_entropy

# --- Test Scaling ---
entropy_list = []
n_list = []

for n in [10, 50, 100, 500, 1000]:
    m_star = 2 * n
    nc2 = mpmath.binomial(n, 2)
    alpha, theta = solve_network_log_corrected(n, m_star)
    if alpha is not None:
        S = calculate_log_corrected_entropy(alpha, theta, nc2)
        entropy_list.append(S)
        n_list.append(n)
        print(f"n={n}: S={S:.4f}, alpha={alpha:.4f}, theta={theta:.4f}")

matplotlib.rcParams.update({'font.size': 28})
plt.figure()
plt.scatter(n_list, entropy_list)
plt.xlabel('n (number of nodes)')
plt.ylabel('Entropy')
plt.title('Entropy vs. n for m* = 2n')
plt.show() 

Trying solver 'secant' for n=10
  [Failed] Solver 'secant' failed with error: Could not find root within given tolerance. (3182830362716757527857563584.8886426636248514967848069545 > 1e-10)
Try another starting point or tweak arguments.
Trying solver 'newton' for n=10
  [Failed] Solver 'newton' failed with error: Could not find root within given tolerance. (3182830362716757527857563584.8886426636248514967848069545 > 1e-10)
Try another starting point or tweak arguments.
Trying solver 'bisect' for n=10
  [Failed] Solver 'bisect' failed with error: Could not find root within given tolerance. (3182830362716757527857563584.8886426636248514967848069545 > 1e-10)
Try another starting point or tweak arguments.
Trying solver 'anderson' for n=10
  [Failed] Solver 'anderson' failed with error: Could not find root within given tolerance. (3182830362716757527857563584.8886426636248514967848069545 > 1e-10)
Try another starting point or tweak arguments.
Trying solver 'muller' for n=10
  [Failed] Solve

KeyboardInterrupt: 

In [3]:
import math
import mpmath

def get_smart_initial_guesses(n, m_star):
    nc2 = math.comb(n, 2)
    
    # Handle edge cases to prevent math domain errors
    if m_star == 0 or m_star == nc2:
        return 2.0, 0.5 
        
    # 1. Calculate target constant C = ln(N choose m*)
    try:
        C = math.log(math.comb(nc2, m_star))
    except ValueError:
        return 2.0, 0.5

    H = C / math.log(C) 
    
    for _ in range(10): 
        if H <= 1.1:    
            H = 1.1
            
        log_H = math.log(H)
        log_log_H = math.log(log_H)
        
        f_H = H * (log_H + log_log_H)
        f_prime_H = log_H + log_log_H + 1.0 + 1.0 / log_H
        
        H = H - (f_H - C) / f_prime_H

    log_H = math.log(H)
    f_prime_H = log_H + math.log(log_H) + 1.0 + 1.0 / log_H
    
    beta_guess = math.log((nc2 - m_star) / m_star) / f_prime_H
    alpha_guess = H - beta_guess * m_star
    
    return alpha_guess, beta_guess

In [2]:
import mpmath
import matplotlib.pyplot as plt
import matplotlib

mpmath.mp.dps = 50  # Set decimal places for high precision

def solve_network_log_corrected(n, m_star):
    nc2 = int(mpmath.binomial(n, 2))
    
    # Adjusted guesses so H > 1, keeping ln(ln(H)) strictly real
    alpha_guess, beta_guess = get_smart_initial_guesses(n, m_star)
    print(f"Initial guesses for n={n}, m*={m_star}: alpha={alpha_guess:.4f}, beta={beta_guess:.4f}")

    def equations(alpha, beta):
        total_prob = 0
        expected_edges = 0
        for m in range(nc2 + 1):
            H = alpha + beta * m
            log_p = -H * (mpmath.log(H) + mpmath.log(mpmath.log(H)))
            
            prob = mpmath.binomial(nc2, m) * mpmath.exp(log_p)
            total_prob += prob
            expected_edges += m * prob

        return [total_prob - 1, expected_edges - m_star]

    def jacobian(alpha, beta):
        J11 = J12 = J22 = 0
        for m in range(nc2 + 1):
            H = alpha + beta * m
            log_H = mpmath.log(H)
            
            # Recompute probability for the derivative weight
            log_p = -H * (log_H + mpmath.log(log_H))
            prob = mpmath.binomial(nc2, m) * mpmath.exp(log_p)

            
            # Calculate the shared derivative multiplier K_m
            K_m = -prob * (log_H + mpmath.log(log_H) + 1 + 1/log_H)
            
            # Accumulate Jacobian entries
            J11 += K_m
            J12 += K_m * m
            J22 += K_m * (m**2)
            
        J21 = J12 # Symmetry shortcut
        
        return [[J11, J12], 
                [J21, J22]]

    solvers = ['mdnewton']
    for solver in solvers:
        print(f"Trying solver '{solver}' for n={n}")
        try:
            kwargs = {'tol': 1e-10, 'maxsteps': 100}
            # Pass the analytical Jacobian directly into findroot
            root = mpmath.findroot(equations, [alpha_guess, beta_guess], solver=solver, J=jacobian, **kwargs)
            print(f"  [Success] Solver '{solver}' converged for m_star={m_star}")
            return float(root[0]), float(root[1])
            
        except Exception as e:
            print(f"  [Failed] Solver '{solver}' failed with error: {e}")
            continue
            
    print(f"  [Error] All solvers failed for m={m_star}")
    return None, None

n = 100
m_star = 2 * n
alpha, beta = solve_network_log_corrected(n, m_star)

if alpha is not None:
    print(f"n={n}: alpha={alpha:.4f}, beta={beta:.4f}")

Initial guesses for n=100, m*=200: alpha=46.6176, beta=0.4140
Trying solver 'mdnewton' for n=100
  [Success] Solver 'mdnewton' converged for m_star=200
n=100: alpha=47.1123, beta=0.4138


In [4]:
import numpy as np
from scipy.optimize import fsolve
from scipy.special import gammaln

def solve_network_scipy(n, m_star):
    nc2 = n * (n - 1) // 2
    m_range = np.arange(nc2 + 1)
    
    # Precompute log of combinations: ln(nCr) = ln(n!) - ln(r!) - ln((n-r)!)
    log_combinations = gammaln(nc2 + 1) - gammaln(m_range + 1) - gammaln(nc2 - m_range + 1)

    def equations(p):
        alpha, beta = p
        H = alpha + beta * m_range
        
        # Ensure H > 1 to keep log(log(H)) real and stable
        H = np.clip(H, 1.000001, None)
        
        # log_p = -H * (ln(H) + ln(ln(H)))
        log_H = np.log(H)
        log_p = -H * (log_H + np.log(log_H))
        
        # Full log-probability: ln(comb * exp(log_p)) = log_comb + log_p
        log_total = log_combinations + log_p
        
        # To prevent underflow/overflow during summation, use the Log-Sum-Exp trick
        max_log = np.max(log_total)
        probs = np.exp(log_total - max_log)
        
        sum_probs = np.sum(probs)
        # Expected value: sum(m * prob) / sum(prob)
        # We need the sum of probabilities to be 1, so we normalize
        expected_m = np.sum(m_range * probs) / sum_probs
        
        # We want:
        # 1. Total prob = 1 (equivalent to max_log + ln(sum_probs) = 0)
        # 2. Expected edges = m_star
        res1 = max_log + np.log(sum_probs) 
        res2 = expected_m - m_star
        
        return [res1, res2]

    # Initial guesses: alpha usually dominates, beta is a small correction
    # For m_star = 2n, H needs to be small but > 1.
    alpha_guess, beta_guess = get_smart_initial_guesses(n, m_star)
    #alpha_guess, beta_guess = 2.0, 0.1
    initial_guess = [alpha_guess, beta_guess]
    
    sol, info, ier, msg = fsolve(equations, initial_guess, full_output=True)
    
    if ier == 1:
        return sol[0], sol[1]
    else:
        print(f"Optimization failed: {msg}")
        return None, None

# Execution
n = 100
m_star = 2 * n
alpha, beta = solve_network_scipy(n, m_star)

if alpha is not None:
    print(f"Results for n={n}:")
    print(f"Alpha: {alpha:.8f}")
    print(f"Beta:  {beta:.8f}")

Results for n=100:
Alpha: 47.11234550
Beta:  0.41380096


In [4]:
import numpy as np
from scipy.optimize import fsolve
from scipy.special import gammaln
import mpmath
import math

mpmath.mp.dps = 100  


#scipy pre solve
def solve_network_scipy(n, m_star):
    nc2 = n * (n - 1) // 2
    
    std_dev = math.sqrt(m_star)
    window_radius = int(15 * std_dev) 
    m_start = max(0, int(m_star) - window_radius)
    m_end = min(nc2, int(m_star) + window_radius)
    
    m_range = np.arange(m_start, m_end + 1)
    log_combinations = gammaln(nc2 + 1) - gammaln(m_range + 1) - gammaln(nc2 - m_range + 1)

    def equations(p):
        alpha, beta = p
        H = np.clip(alpha + beta * m_range, 1.000001, None)
        log_H = np.log(H)
        log_p = -H * (log_H + np.log(log_H))
        
        log_total = log_combinations + log_p
        max_log = np.max(log_total)
        probs = np.exp(log_total - max_log)
        
        sum_probs = np.sum(probs)
        expected_m = np.sum(m_range * probs) / sum_probs
        
        return [max_log + np.log(sum_probs), expected_m - m_star]

    alpha_guess, beta_guess = get_smart_initial_guesses(n, m_star)
    
    sol, info, ier, msg = fsolve(equations, [alpha_guess, beta_guess], full_output=True)
    if ier == 1:
        return sol[0], sol[1]
    return None, None

#mpmath high precision solve with windowing
def solve_network_log_corrected_fast(n, m_star):
    nc2 = int(mpmath.binomial(n, 2))
    
    alpha_guess, beta_guess = solve_network_scipy(n, m_star)
    if alpha_guess is None:
        print("SciPy pre-solver failed.")
        alpha_guess, beta_guess = get_smart_initial_guesses(n, m_star)
        
    print(f"Initial Guesses: alpha={alpha_guess:.8f}, beta={beta_guess:.8f}")

    std_dev = math.sqrt(m_star)
    window_radius = int(15 * std_dev) 
    m_start = max(0, int(m_star) - window_radius)
    m_end = min(nc2, int(m_star) + window_radius)
    
    print(f"Summation Window: m in [{m_start}, {m_end}] (Total terms: {m_end - m_start + 1} instead of {nc2 + 1})")

    #pre calc nc2cm for window
    window_combinations = {m: mpmath.binomial(nc2, m) for m in range(m_start, m_end + 1)}

    def equations(alpha, beta):
        total_prob = mpmath.mpf(0)
        expected_edges = mpmath.mpf(0)
        
        for m in range(m_start, m_end + 1):
            H = alpha + beta * m
            log_H = mpmath.log(H)
            log_p = -H * (log_H + mpmath.log(log_H))
            
            prob = window_combinations[m] * mpmath.exp(log_p)
            total_prob += prob
            expected_edges += m * prob

        return [total_prob - 1, expected_edges - m_star]

    def jacobian(alpha, beta):
        J11 = J12 = J22 = mpmath.mpf(0)
        
        for m in range(m_start, m_end + 1):
            H = alpha + beta * m
            log_H = mpmath.log(H)
            log_p = -H * (log_H + mpmath.log(log_H))
            
            prob = window_combinations[m] * mpmath.exp(log_p)
            
            K_m = -prob * (log_H + mpmath.log(log_H) + 1 + 1/log_H)
            
            J11 += K_m
            J12 += K_m * m
            J22 += K_m * (m**2)
            
        return [[J11, J12], 
                [J12, J22]]


    try:
        kwargs = {'tol': 1e-45, 'maxsteps': 20} # Needs fewer steps because guess is great
        root = mpmath.findroot(equations, [alpha_guess, beta_guess], solver='mdnewton', J=jacobian, **kwargs)
        return float(root[0]), float(root[1]), root
    except Exception as e:
        print(f"  [Failed] mpmath solver error: {e}")
        return None, None, None

n = 1000
m_star = 2 * n
alpha, beta, high_prec_root = solve_network_log_corrected_fast(n, m_star)

if alpha is not None:
    print("\nFinal High-Precision Results")
    print(f"alpha = {high_prec_root[0]}")
    print(f"beta =  {high_prec_root[1]}")

Initial Guesses: alpha=348.08997117, beta=0.53186411
Summation Window: m in [1330, 2670] (Total terms: 1341 instead of 499501)

Final High-Precision Results
alpha = 348.0899711660607876592104862211052600449741051517549108135884812640594254178314884550778637149731301
beta =  0.5318641135990459909666865173087460172029796410400425526952183660427268493643839198148351589925312716


In [16]:
def verify_solution(alpha, beta, n, m_star):
    nc2 = int(mpmath.binomial(n, 2))
    total_prob = mpmath.mpf(0)
    expected_edges = mpmath.mpf(0)
    
    for m in range(nc2 + 1):
        if (m%(nc2//100)==0): print(m)
        H = alpha + beta * m
        log_H = mpmath.log(H)
        log_p = -H * (log_H + mpmath.log(log_H))
        
        prob = mpmath.binomial(nc2, m) * mpmath.exp(log_p)
        total_prob += prob
        expected_edges += m * prob
    
    print(f"Verification: Total Probability = {total_prob}, Expected Edges = {expected_edges}"
          f" (Target: 1 and {m_star})")


In [17]:
verify_solution(alpha, beta, n, m_star)

0
4995
9990
14985
19980
24975
29970
34965
39960
44955
49950
54945
59940
64935
69930
74925
79920
84915
89910
94905
99900
104895
109890
114885
119880
124875
129870
134865
139860
144855
149850
154845
159840
164835
169830
174825
179820
184815
189810
194805
199800
204795
209790
214785
219780
224775
229770
234765
239760
244755
249750
254745
259740
264735
269730
274725
279720
284715
289710
294705
299700
304695
309690
314685
319680
324675
329670
334665
339660
344655
349650
354645
359640
364635
369630
374625
379620
384615
389610
394605
399600
404595
409590
414585
419580
424575
429570
434565
439560
444555
449550
454545
459540
464535
469530
474525
479520
484515
489510
494505
499500
Verification: Total Probability = 1.00000000000110593261685994128116018138083408131329760064386989001000303679427338665862253445568931, Expected Edges = 2000.00000000221181933298409256081303483398376432700367969706736304558519459756292287654746720516783 (Target: 1 and 2000)


In [5]:
def calculate_log_corrected_entropy(alpha, beta, nc2):
    nc2 = int(nc2)
    total_entropy = mpmath.mpf(0)
    std_dev = math.sqrt(m_star)
    window_radius = int(15 * std_dev) 
    m_start = max(0, int(m_star) - window_radius)
    m_end = min(nc2, int(m_star) + window_radius)

    #pre calc nc2cm for window
    window_combinations = {m: mpmath.binomial(nc2, m) for m in range(m_start, m_end + 1)}

    for m in range(m_start, m_end + 1):
        H = alpha + beta * m
        log_H = mpmath.log(H)
        log_p = -H * (log_H + mpmath.log(log_H))
        p = mpmath.exp(log_p)

        term = -p * log_p / mpmath.log(-log_p)

        total_entropy += window_combinations[m] * term
        
    return total_entropy

In [10]:
entropy_list = []
n_list = [10, 50, 100, 500, 1000, 5000, 10000, 50000]
for n in n_list:
    m_star = 2 * n
    nc2 = mpmath.binomial(n, 2)
    alpha, beta, _ = solve_network_log_corrected_fast(n, m_star)
    if alpha is not None:
        S = calculate_log_corrected_entropy(alpha, beta, nc2)
        entropy_list.append(S)
        print(f"n={n}: S={S}, alpha={alpha}, beta={beta}")
print(entropy_list)

Initial Guesses: alpha=8.92141738, beta=0.04894986
Summation Window: m in [0, 45] (Total terms: 46 instead of 46)
n=10: S=9.009184782796779996830347596416495580170990100157022738410596588653786237411061516804774913575373713, alpha=8.921417381806465, beta=0.04894985529205684
Initial Guesses: alpha=26.74474999, beta=0.35620868
Summation Window: m in [0, 250] (Total terms: 251 instead of 1226)
n=50: S=59.21810839472417305439446387647089398977226211574980643799868321639894143251470196525702358750506962, alpha=26.744749991531865, beta=0.3562086806937397
Initial Guesses: alpha=47.11234550, beta=0.41380096
Summation Window: m in [0, 412] (Total terms: 413 instead of 4951)
n=100: S=124.4406861302789635741160092416279520738034898181139346472565127256441830155776793513837421705426969, alpha=47.11234549996069, beta=0.4138009552706302
Initial Guesses: alpha=188.23446034, beta=0.50347701
Summation Window: m in [526, 1474] (Total terms: 949 instead of 124751)
n=500: S=671.570342164005023517364250744

In [11]:
entropy_list = [float(S) for S in entropy_list]
print(entropy_list)

[9.00918478279678, 59.218108394724176, 124.44068613027896, 671.5703421640051, 1375.8319488775946, 7191.159330674699, 14607.705483877451, 75288.4041092324]


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams.update({'font.size': 28})
plt.figure()
plt.scatter(n_list, entropy_list)
plt.xlabel('n (number of nodes)')
plt.ylabel('Entropy')
plt.title('Entropy vs. n for m* = 2n')
plt.show() 